In [0]:
from pyspark.sql import functions as F

df_mio   = spark.table("catalog_cemm_expl_bcp_prod.bcp_expl_007.bhv_troncal_cliente_base_v2_dev").filter(F.col("codmes")==202512)
df_arnold = spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_copy").filter(F.col("codmes")==202512)

j = (df_mio.select("codclaveunicocli","codclavepartycli",
                   F.col("ctddiaatraso").alias("mio"),
                   F.col("flg_tc").alias("flg_tc_mio"),
                   F.col("flg_cef").alias("flg_cef_mio"),
                   F.col("flg_veh").alias("flg_veh_mio"),
                   F.col("flg_hip").alias("flg_hip_mio"))
     .join(df_arnold.select("codclaveunicocli",
                            F.col("ctddiaatraso").alias("arnold")),
           "codclaveunicocli"))

# Patrón de la diferencia
j.filter(F.col("mio") != F.col("arnold")).groupBy(
    "flg_tc_mio","flg_cef_mio","flg_veh_mio","flg_hip_mio"
).agg(
    F.count("*").alias("n"),
    F.avg(F.col("mio")).alias("avg_mio"),
    F.avg(F.col("arnold")).alias("avg_arnold"),
    F.sum(F.when(F.col("mio")==0, 1).otherwise(0)).alias("mio_es_0"),
    F.sum(F.when(F.col("arnold")==0, 1).otherwise(0)).alias("arnold_es_0"),
).orderBy(F.desc("n")).show(20, truncate=False)

# Muestra de los que difieren (enfocado en mio=0 vs arnold>0)
j.filter((F.col("mio")==0) & (F.col("arnold")>0)).select(
    "codclavepartycli","mio","arnold",
    "flg_tc_mio","flg_cef_mio","flg_veh_mio","flg_hip_mio"
).show(20)